In [2]:
import json
import os
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

PROCESSED_PATH = Path("../docs/processed")
CHROMA_PATH = Path("../chroma_db")

client_groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("Groq client ready.")

Groq client ready.


In [3]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client_chroma = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client_chroma.get_collection(
    name="wso2_apim_docs",
    embedding_function=embedding_fn
)

print(f"Connected to collection: {collection.count()} chunks")

/Users/heshangamage/ws02-devassist/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20251.89it/s]


Connected to collection: 3127 chunks


In [4]:
def retrieve(query, n_results=5, folder_filter=None):
    """
    Retrieve the most relevant chunks for a query.
    Optionally filter by folder (e.g. only search api-security).
    """
    where = {"folder": folder_filter} if folder_filter else None
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )
    
    chunks = []
    for i in range(len(results['documents'][0])):
        chunks.append({
            'text': results['documents'][0][i],
            'page_title': results['metadatas'][0][i]['page_title'],
            'section': results['metadatas'][0][i]['section'],
            'folder': results['metadatas'][0][i]['folder'],
            'source_file': results['metadatas'][0][i]['source_file'],
            'distance': results['distances'][0][i]
        })
    
    return chunks

In [5]:
def build_prompt(question, retrieved_chunks):
    """
    Build the LLM prompt by combining the question
    with the retrieved documentation chunks as context.
    """
    
    # Format retrieved chunks as context
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks):
        context_parts.append(
            f"--- Source {i+1}: {chunk['page_title']} > {chunk['section']} ---\n"
            f"{chunk['text']}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a helpful technical assistant for WSO2 API Manager. 
Answer the developer's question using ONLY the provided documentation excerpts.

RULES:
- Be precise and technical
- Include code examples when relevant and when the docs contain them
- If the docs don't contain enough information to fully answer, say so clearly
- Always mention which section of the docs your answer comes from
- Format code blocks with the correct language tag (```python, ```yaml, ```bash etc)
- Keep your answer focused and practical

DOCUMENTATION EXCERPTS:
{context}

DEVELOPER QUESTION:
{question}

ANSWER:"""
    
    return prompt

In [6]:
def generate_answer(question, retrieved_chunks):
    """
    Generate an answer using the LLM given the question
    and retrieved context chunks.
    """
    prompt = build_prompt(question, retrieved_chunks)
    
    response = client_groq.chat.completions.create(
        model=os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant"),
        max_tokens=1000,
        temperature=0.1,  # low temperature for factual technical answers
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content.strip()

In [7]:
def ask(question, n_chunks=5, folder_filter=None, verbose=False):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Build prompt with context
    3. Generate answer
    4. Return answer with sources
    """
    
    # Step 1 — Retrieve
    chunks = retrieve(question, n_results=n_chunks, folder_filter=folder_filter)
    
    if verbose:
        print(f"Retrieved {len(chunks)} chunks:")
        for c in chunks:
            print(f"  [{c['distance']:.3f}] {c['page_title']} > {c['section']}")
        print()
    
    # Step 2 & 3 — Generate
    answer = generate_answer(question, chunks)
    
    # Step 4 — Package result
    sources = list({
        f"{c['page_title']} > {c['section']}"
        for c in chunks
    })
    
    return {
        'question': question,
        'answer': answer,
        'sources': sources,
        'chunks_used': len(chunks)
    }

In [8]:
# Test 1 — Basic concept question
result = ask(
    "What is WSO2 API Manager and what are its main features?",
    verbose=True
)
print("ANSWER:")
print(result['answer'])
print("\nSOURCES:")
for s in result['sources']:
    print(f"  • {s}")

Retrieved 5 chunks:
  [0.087] Overview > Overview
  [0.157] About this Release > About this Release
  [0.185] WSO2's Centralized API Management: The Single Control Plane for Multiple Gateways > General
  [0.201] Key Concepts > General
  [0.205] Architecture and Key Components > Tooling

ANSWER:
WSO2 API Manager is a comprehensive, fully open-source API management platform that provides complete API lifecycle management with enterprise-grade security, advanced gateway capabilities, and AI-powered features. Its main features include:

* Design, secure, publish, and analyze APIs
* Provide developers with a rich marketplace experience
* Enterprise-grade security
* Advanced gateway capabilities
* AI-powered features

(From: Overview > Overview)

This platform enables organizations to drive their digital transformation strategy by building, integrating, and exposing digital services as managed APIs in the cloud, on-premise, and hybrid architectures. (From: About this Release > About this Rel

In [9]:
# Test 2 — Technical how-to question
result = ask(
    "How do I secure an API using OAuth2 in WSO2 API Manager?",
    verbose=True
)
print("ANSWER:")
print(result['answer'])
print("\nSOURCES:")
for s in result['sources']:
    print(f"  • {s}")

Retrieved 5 chunks:
  [0.168] Secure Endpoint with OAuth 2.0 > Securing an endpoint with OAuth 2.0 in WSO2 API Manager
  [0.183] Securing APIs using OAuth2 Access Tokens > General
  [0.212] Securing APIs using OAuth2 Access Tokens > Securing APIs using OAuth2 Access Tokens
  [0.213] Secure APIs with API Keys > Secure APIs with API Keys
  [0.217] Overview > Overview

ANSWER:
To secure an API using OAuth2 in WSO2 API Manager, follow these steps:

1. Click **Endpoints** in API Publisher.
2. Click the Endpoint Security symbol that corresponds to the endpoint that you want to secure with OAuth 2.0.
3. Click on OAuth 2.0 from the drop-down menu.
4. Select the preferred grant-type from the next drop-down menu and enter the required properties.

Specifically, for Client Credentials, provide the following properties:

```python
Token URL
This is the URL for the token endpoint of the OAuth-protected backend server.
```

After securing the endpoint, you need to redeploy the API. To do this, sign 

In [10]:
# Test 3 — MCP specific question (most relevant to WSO2's current focus)
result = ask(
    "What is the MCP Gateway in WSO2 and how does it work?",
    verbose=True
)
print("ANSWER:")
print(result['answer'])
print("\nSOURCES:")
for s in result['sources']:
    print(f"  • {s}")

Retrieved 5 chunks:
  [0.269] Architecture and Key Components > General
  [0.285] Getting Started with MCP Gateway > Next Step → MCP Server Creation Options
  [0.309] Getting Started with MCP Gateway > Getting Started with MCP Gateway
  [0.311] WSO2 AI Gateway > MCP Gateway
  [0.311] WSO2 AI Gateway > MCP Gateway

ANSWER:
**MCP Gateway Overview**

The MCP Gateway is a component of WSO2 API Manager that transforms existing APIs into AI-ready tools that AI agents can discover and invoke. It is built on the JSON-RPC–based Model Context Protocol specification and standardizes how applications interact with Large Language Models (LLMs) by exposing callable tools that AI agents can invoke in workflows.

**Architecture**

The MCP Gateway implements a three-tier architecture:

- **Host**: Runtime where the MCP client operates, such as an AI agent or gateway
- **Client**: Mediates communication with MCP servers
- **Servers**: Publish tools, schemas, and metadata for discovery and invocation

**

In [11]:
# Test 4 — Code generation question
result = ask(
    "Show me how to invoke a WSO2 API using Python with OAuth2 authentication",
    verbose=True
)
print("ANSWER:")
print(result['answer'])
print("\nSOURCES:")
for s in result['sources']:
    print(f"  • {s}")

Retrieved 5 chunks:
  [0.271] Client Credentials Grant > Invoking the Token API to generate the tokens
  [0.300] Provisioning Out-of-Band OAuth2 Clients > Step 3 - Provision the out-of-band OAuth2 client
  [0.320] Obtaining User Profile Information with OpenID Connect > General
  [0.320] Obtaining User Profile Information with OpenID Connect > General
  [0.339] Securing APIs using OAuth2 Access Tokens > Securing APIs using OAuth2 Access Tokens

ANSWER:
To invoke a WSO2 API using Python with OAuth2 authentication, you can use the `requests` library along with the `requests-oauthlib` library to handle OAuth2 authentication.

First, install the required libraries using pip:

```bash
pip install requests requests-oauthlib
```

Then, you can use the following code snippet as an example to invoke a WSO2 API using OAuth2 authentication:

```python
import requests
from requests_oauthlib import OAuth2Session

# Client ID and Client Secret
client_id = "your_client_id"
client_secret = "your_clien